## Lecture 2: Computer Architecture and Memory

**After EACH milestone:**
1. **Share with neighbor (compare approaches)**
2. **Take a 10-minute break (self-managed)**
3. **Continue to next milestone**

**How it works:**
1. **After each successful optimization, click “Add entry” in the Performance Tracker on Moodle**
2. **Fill in your implementation type, resolution, and median runtime**
3. **Browse others’ entries — are you in the right ballpark?**

****Benchmarking standard**: Report the median of **$\ge3$** runs using `time.perf_counter()`. Default resolution: $1024\times1024$.**

****Today’s task:** After Milestone 2, log your **naive baseline** and your NumPy vectorized result. Your naive entry is the most important one — it’s your starting point!**

### ****Milestone 1:** Basic arrays & meshgrid (Target: 15 min)**

****Your Task:** Create the complex grid C for the Mandelbrot set**

- **Create 1D arrays with `linspace`**

In [1]:
import numpy as np

# Copied from slide 25
x = np.linspace(-2, 1, 1024)     # 1024 x - values
y = np.linspace(-1.5, 1.5, 1024)  # 1024 y - values

- **Create 2D grid with `meshgrid`**

In [2]:
# Copied from slide 25

X, Y = np.meshgrid(x, y)
# X , Y are now 1024 x1024 arrays

- **Combine into complex array `C`**

In [3]:
# Copied from slide 25

C = X + 1j * Y  # Note: 1j is imaginary unit in Python
# C is now 1024x1024 complex array

- **Verify shape and dtype**

In [4]:
print(f"Shape: {C.shape}")
print(f"Type: {C.dtype}")

Shape: (1024, 1024)
Type: complex128


****Done?** Commit → share with neighbor → break → Milestone 2**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git add l02_exercises.ipynb 
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git commit -m "l02: milestone 1 done"
[main 806b5d0] l02: milestone 1 done
 1 file changed, 489 insertions(+)
 create mode 100644 l02_exercises.ipynb
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit 806b5d0e5cc8edf89cabd0725f15f1fbd2729107 (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Mon Mar 2 21:34:41 2026 +0100

    l02: milestone 1 done
```

### ****Milestone 2:** Full vectorized Mandelbrot (Target: 40 min)**

****Your Task:** Replace Python loops with NumPy operations**

****Naive approach has 3 nested loops:****
1. **Loop over rows (`i`)**
2. **Loop over columns (`j`) → selects pixel/point $c$**
3. **Loop over iterations (`n`) → computes $z = z^2 + c$ until escape**

**Hints: `np.zeros like(C)`, `np.zeros(C.shape, dtype=int)`**

#### ****NumPy approach — eliminate loops 1 & 2:****

- **Initialize `Z` and `M` arrays (same shape as `C`)**
- **Keep only loop 3 (iterations), operate on *all* pixels at once**
- **Boolean mask: `mask = np.abs(Z) <= 2`**
- **Update only unescaped points: `Z[mask] = Z[mask]**2 + C[mask]`**
- **Increment iteration count: `M[mask] += 1`**

In [5]:
%pycat mandelbrot.py

# Template from slide 30, lecture 01
"""
Mandelbrot Set Generator

Author: Mikkel Korsgaard Sørensen
Course: Numerical Scientific Computing 2026
"""
import numpy as np
# This is a comment

def f(x):
    """
    Example function.

    Parameters
    ----------
    x : float
        Input value

    Returns
    -------
    float
        Output value
    """
    # TODO: Implement the algorithm
    pass


def mandelbrot_point(c: complex, max_iter: int) -> int:
    z = 0j
    for n in range(max_iter):
        if abs(z) > 2:
            return n
        z = z**2 + c

    return max_iter


def compute_mandelbrot(
    x_min: float,
    x_max: float,
    y_min: float,
    y_max: float,
    width: int,
    height: int,
    max_iter: int
) -> list[list[int]]:
    x = np.linspace(x_min, x_max, width)
    y = np.linspace(y_min, y_max, height)

    img = []

    for i in range(height):
        img_row = []
        for j in range(width):
            img_row.append(mandelbrot_point(x[j] + 1j * y[i], m

****Done?** Commit → log in Performance Tracker → share with neighbor**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit 069bb1e23fb1d6e9aa03a710ad0af096ad5b78e1 (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Mon Mar 2 20:39:57 2026 +0000

    l02: milestone 2 done
```

**Validating Your Results**

In [6]:
from mandelbrot import compute_mandelbrot, compute_mandelbrot_numpy

conf = {
    "x_min": -2,
    "x_max": 1,
    "y_min": -1.5,
    "y_max": 1.5,
    "width": 1024,
    "height": 1024,
    "max_iter": 100
}
naive_result = compute_mandelbrot(**conf)
numpy_result = compute_mandelbrot_numpy(**conf)

# From slide 32
# CORRECT - Use np.allclose():
if np.allclose(naive_result, numpy_result):
    print("Results match!")
else:
    print("Results differ!")

# Check where they differ :
diff = np.abs(naive_result - numpy_result)
print(f"Max difference: {diff.max()}")
print(f"Different pixels: {( diff > 0).sum()}")

Results match!
Max difference: 0
Different pixels: 0


##### After Milestone 2, log your **naive baseline** and your **NumPy vectorized** result.

In [7]:
# From slide 29
import time
import statistics


def benchmark(func, *args, n_runs=3):
    """Time func, return median of n_runs."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        result = func(*args)
        times.append(time.perf_counter() - t0)

    median_t = statistics.median(times)
    print(f"Median: {median_t:.4f}s"
          f"(min={min(times):.4f}, max={max(times):.4f})")
    return median_t, result

In [8]:
from mandelbrot import compute_mandelbrot as naive_python
from mandelbrot import compute_mandelbrot_numpy as numpy_python

# Mostly copied from slide 29
t1, M1 = benchmark(naive_python, -2, 1, -1.5, 1.5, 1024, 1024, 100)
t2, M2 = benchmark(numpy_python, -2, 1, -1.5, 1.5, 1024, 1024, 100)

print(f"Naïve Python (median): {t1:.3f}s")
print(f"Numpy Python (median): {t2:.3f}s. Speedup of {t1/t2:.2f}x")

Median: 4.0968s(min=4.0420, max=4.1096)
Median: 0.5983s(min=0.5888, max=0.6060)
Naïve Python (median): 4.097s
Numpy Python (median): 0.598s. Speedup of 6.85x


### ****Milestone 3:** Memory access pattern analysis (Target: 15 min)**

****Your Task:** Measure the effect of memory layout on performance**

**1. Create a large **square** array: `A = np.random.rand(N, N)` with $N = 10000$**

In [9]:
N = 10_000
A = np.random.rand(N, N)

**2. Write a function that computes **row sums** by looping over rows:**
- **`for i in range(N): s = np.sum(A[i, :])`**

In [10]:
def row_sum(A, N):
    for i in range(N):
        s = np.sum(A[i, :])

**3. Write a function that computes **column sums** by looping over columns:**
- **`for j in range(N): s = np.sum(A[:, j])`**

In [11]:
def column_sum(A, N):
    for j in range(N):
        s = np.sum(A[:, j])

**4. Time both. Both loops run $N$ times — which is faster and why?**

In [12]:
t_row, _ = benchmark(row_sum, A, N)
t_col, _ = benchmark(column_sum, A, N)

print(f"Row sum (median): {t_row:.3f}s")
print(f"Column sum (median): {t_col:.3f}s. Slowdown of {t_col/t_row:.2f}x")

Median: 0.0789s(min=0.0770, max=0.0830)
Median: 0.1532s(min=0.1522, max=0.1564)
Row sum (median): 0.079s
Column sum (median): 0.153s. Slowdown of 1.94x


**5. Now try with `A_f = np.asfortranarray(A)` (column-major). What changes?**

In [13]:
A_f = np.asfortranarray(A)

t_row, _ = benchmark(row_sum, A_f, N)
t_col, _ = benchmark(column_sum, A_f, N)

print(f"Row sum (median): {t_row:.3f}s")
print(f"Column sum (median): {t_col:.3f}s. Speedup of {t_row/t_col:.2f}x")

Median: 0.1543s(min=0.1512, max=0.1742)
Median: 0.0689s(min=0.0648, max=0.0699)
Row sum (median): 0.154s
Column sum (median): 0.069s. Speedup of 2.24x


****Done?** Commit → share results with neighbor → Milestone 4**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit 396437f33a71bec972e10695b289c652c6c016de (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Mon Mar 2 20:44:24 2026 +0000

    l02: milestone 3 done
```

### ****Milestone 4:** Problem size scaling (Target: 15 min)**

****Your Task:** How does Mandelbrot runtime scale with grid size?**

**1. Run your vectorized Mandelbrot for grid sizes: $256, 512, 1024, 2048, 4096$**

**2. Record runtime for each (use `timeit` or manual timing)**

**3. Plot: grid size vs. runtime**

**4. **Predict:** If $1024 \times 1024$ takes $X$ seconds, what should $2048 \times 2048$ take?**
- **(Hint: $4\times$ the pixels — do you get $4\times$ the time?)**

**Questions to consider:**

- **Is the scaling linear in number of pixels?**

- **At what size does the working set exceed your L3 cache?**

- **Do you see a “knee” in the scaling curve where performance degrades?**

****Done?** Commit → share plot with neighbor → try extensions**

### ****Extensions** Memory profiling, region exploration (no time limit)**

#### ****Extension 1: Memory Profiling****

- **Install: `mamba install memory_profiler`**

- **Compare peak memory usage: naive vs NumPy**

- **Why does the vectorized version use more memory?**

#### ****Extension 2: Explore Different Mandelbrot Regions****

- **Zoom into the boundary region (more iterations needed)**

- **Does runtime change? Why? (Hint: the mask)**

- **Try the famous named regions: **Seahorse Valley**, **Elephant Valley**, or a **Deep Seahorse Spiral****

- ***Coordinates for these regions are on the last backup slide***

#### ****Extension 3: Help Others!****

- **Teaching deepens your understanding**